<a href="https://www.kaggle.com/code/gunjan9960/credit-card-fraud-pm-analysis?scriptVersionId=317030032" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kartik2112/fraud-detection/fraudTest.csv
/kaggle/input/datasets/kartik2112/fraud-detection/fraudTrain.csv


# Credit Card Fraud Detection — PM Analysis
## Project goal
Build a model that predicts whether a transaction is fraudulent.
Extract which signals predict fraud and translate into product decisions.

## Key PM question
Which transaction signals matter most — and what does that tell us about how to build a better fraud detection system?

In [2]:
import pandas as pd
import numpy as np

# Load both files
train = pd.read_csv('/kaggle/input/datasets/kartik2112/fraud-detection/fraudTrain.csv')
test  = pd.read_csv('/kaggle/input/datasets/kartik2112/fraud-detection/fraudTest.csv')

# Basic facts
print("=== TRAINING DATA ===")
print(f"Total transactions: {len(train):,}")
print(f"Columns: {list(train.columns)}")
print(f"\nFraud rate: {train['is_fraud'].mean():.2%}")
print(f"Fraudulent transactions: {train['is_fraud'].sum():,}")
print(f"Legitimate transactions: {(train['is_fraud']==0).sum():,}")

print("\n=== TEST DATA ===")
print(f"Total transactions: {len(test):,}")
print(f"Fraud rate: {test['is_fraud'].mean():.2%}")

print("\n=== FIRST 3 ROWS ===")
train.head(3)

=== TRAINING DATA ===
Total transactions: 1,296,675
Columns: ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']

Fraud rate: 0.58%
Fraudulent transactions: 7,506
Legitimate transactions: 1,289,169

=== TEST DATA ===
Total transactions: 555,719
Fraud rate: 0.39%

=== FIRST 3 ROWS ===


,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0


## Step 2 — Understanding the data
Key observations:
- 1.3M training transactions, 0.58% fraud rate
- Class imbalance: 99.42% legitimate vs 0.58% fraud
- Rich readable features: merchant, category, amount, location
- Key insight: distance between cardholder and merchant will be our engineered feature

In [3]:
# PM Question 1: Which categories have highest fraud rates?
print("=== FRAUD RATE BY CATEGORY ===")
category_fraud = train.groupby('category')['is_fraud'].agg(['mean','sum','count'])
category_fraud.columns = ['fraud_rate','fraud_count','total']
category_fraud['fraud_rate_pct'] = (category_fraud['fraud_rate']*100).round(2)
print(category_fraud.sort_values('fraud_rate', ascending=False).to_string())

print("\n=== FRAUD RATE BY GENDER ===")
print(train.groupby('gender')['is_fraud'].mean().round(4))

print("\n=== TRANSACTION AMOUNT: FRAUD VS LEGITIMATE ===")
print("Fraudulent transactions - avg amount: $", round(train[train['is_fraud']==1]['amt'].mean(),2))
print("Legitimate transactions - avg amount: $", round(train[train['is_fraud']==0]['amt'].mean(),2))
print("Fraudulent transactions - median amount: $", round(train[train['is_fraud']==1]['amt'].median(),2))
print("Legitimate transactions - median amount: $", round(train[train['is_fraud']==0]['amt'].median(),2))

=== FRAUD RATE BY CATEGORY ===
                fraud_rate  fraud_count   total  fraud_rate_pct
category                                                       
shopping_net      0.017561         1713   97543            1.76
misc_net          0.014458          915   63287            1.45
grocery_pos       0.014098         1743  123638            1.41
shopping_pos      0.007225          843  116672            0.72
gas_transport     0.004694          618  131659            0.47
misc_pos          0.003139          250   79655            0.31
grocery_net       0.002948          134   45452            0.29
travel            0.002864          116   40507            0.29
entertainment     0.002478          233   94014            0.25
personal_care     0.002424          220   90758            0.24
kids_pets         0.002114          239  113035            0.21
food_dining       0.001651          151   91461            0.17
home              0.001608          198  123115            0.16
health_fi

## Step 2 — Key insights from data exploration

**Finding 1 — Online shopping is highest fraud category**
shopping_net: 1.76% fraud rate vs health_fitness: 0.15%
10x more fraud online vs in-person. Apply stricter verification to online merchants.

**Finding 2 — Gender is not a useful signal**
Female: 0.53% vs Male: 0.64% — nearly identical. Exclude from model features.

**Finding 3 — Fraudulent transactions are 8x larger**
Avg fraud: $531 vs avg legitimate: $67
Fraudsters make large purchases quickly. Amount = strong feature.

In [4]:
import pandas as pd
import numpy as np

def engineer_features(df):

    # Feature 1: Transaction hour (fraud spikes at night)
    df['trans_hour'] = pd.to_datetime(df['trans_date_trans_time']).dt.hour

    # Feature 2: Transaction day of week
    df['trans_dayofweek'] = pd.to_datetime(df['trans_date_trans_time']).dt.dayofweek

    # Feature 3: Distance between cardholder and merchant
    # Using simple Euclidean distance on lat/long
    df['distance_km'] = np.sqrt(
        (df['lat'] - df['merch_lat'])**2 +
        (df['long'] - df['merch_long'])**2
    ) * 111  # rough conversion to km

    # Feature 4: Cardholder age
    df['age'] = 2024 - pd.to_datetime(df['dob']).dt.year

    return df

train = engineer_features(train)
test  = engineer_features(test)

# PM check: does distance differ between fraud and legit?
print("=== DISTANCE: FRAUD VS LEGITIMATE ===")
print(f"Avg distance - Fraudulent:  {train[train['is_fraud']==1]['distance_km'].mean():.1f} km")
print(f"Avg distance - Legitimate:  {train[train['is_fraud']==0]['distance_km'].mean():.1f} km")

# PM check: which hours have most fraud?
print("\n=== FRAUD BY HOUR OF DAY ===")
hourly = train.groupby('trans_hour')['is_fraud'].mean().round(4)
print(hourly.sort_values(ascending=False).head(8))

print("\nFeatures engineered successfully!")

=== DISTANCE: FRAUD VS LEGITIMATE ===
Avg distance - Fraudulent:  85.2 km
Avg distance - Legitimate:  85.0 km

=== FRAUD BY HOUR OF DAY ===
trans_hour
22    0.0288
23    0.0284
1     0.0153
0     0.0149
2     0.0147
3     0.0142
5     0.0014
7     0.0013
Name: is_fraud, dtype: float64

Features engineered successfully!


## Step 3 — Feature engineering insights

**Finding 4 — Distance: hypothesis rejected**
Fraud distance (85.2km) ≈ Legitimate distance (85.0km)
Distance is not useful in this dataset. Dropping it as a feature.
PM lesson: always validate assumptions with data before building.

**Finding 5 — Time of day is a strong signal**
Hour 22-23 (10-11pm): ~2.86% fraud rate
Hour 7 (7am): 0.13% fraud rate — 22x difference
Fraudsters operate at night. Apply stricter checks to late-night transactions.

In [5]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score, confusion_matrix

# Select features
feature_cols = [
    'amt',           # transaction amount
    'category',      # merchant category
    'trans_hour',    # hour of day
    'trans_dayofweek', # day of week
    'age',           # cardholder age
    'city_pop',      # city population
]

# Encode category (text → numbers)
le = LabelEncoder()
train['category_enc'] = le.fit_transform(train['category'])
test['category_enc']  = le.transform(test['category'])

feature_cols = ['amt','category_enc','trans_hour',
                'trans_dayofweek','age','city_pop']

X_train = train[feature_cols]
y_train = train['is_fraud']
X_test  = test[feature_cols]
y_test  = test['is_fraud']

print(f"Training on {len(X_train):,} transactions...")
print(f"Features: {feature_cols}")

# Train — scale_pos_weight handles class imbalance
# 99.42% legit vs 0.58% fraud → weight = 1289169/7506 ≈ 171
fraud_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"\nClass imbalance weight: {fraud_weight:.0f}x")

model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=fraud_weight,  # KEY: tells model fraud is rare
    eval_metric='auc',
    random_state=42
)

model.fit(X_train, y_train, verbose=False)
print("Model trained!")

Training on 1,296,675 transactions...
Features: ['amt', 'category_enc', 'trans_hour', 'trans_dayofweek', 'age', 'city_pop']

Class imbalance weight: 172x
Model trained!


In [6]:
from sklearn.metrics import (classification_report, roc_auc_score, 
                              f1_score, confusion_matrix, precision_score,
                              recall_score)
import pandas as pd

# Get predictions
y_pred      = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:,1]

# Core metrics
auc = roc_auc_score(y_test, y_pred_prob)
f1  = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)

print("="*50)
print("MODEL PERFORMANCE")
print("="*50)
print(f"AUC:       {auc:.3f}  (target: >0.90 for fraud)")
print(f"F1 Score:  {f1:.3f}  (target: >0.60)")
print(f"Precision: {precision:.3f}  (of flagged txns, how many are real fraud?)")
print(f"Recall:    {recall:.3f}  (of all fraud, how many did we catch?)")

# Confusion matrix — the most important PM table
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("\n" + "="*50)
print("CONFUSION MATRIX — PM TRANSLATION")
print("="*50)
print(f"True Negatives  (correctly said NOT fraud): {tn:,}")
print(f"False Positives (wrongly blocked legit txn): {fp:,}")
print(f"False Negatives (missed actual fraud):       {fn:,}")
print(f"True Positives  (correctly caught fraud):    {tp:,}")

print("\n" + "="*50)
print("BUSINESS IMPACT")
print("="*50)
total_fraud = y_test.sum()
print(f"Total fraudulent transactions in test: {total_fraud:,}")
print(f"Fraud caught:   {tp:,} ({tp/total_fraud:.1%})")
print(f"Fraud missed:   {fn:,} ({fn/total_fraud:.1%})")
print(f"Customers wrongly blocked: {fp:,}")
avg_fraud_amt = test[test['is_fraud']==1]['amt'].mean()
print(f"\nAvg fraud amount: ${avg_fraud_amt:.2f}")
print(f"Estimated fraud caught value: ${tp * avg_fraud_amt:,.0f}")
print(f"Estimated fraud missed value: ${fn * avg_fraud_amt:,.0f}")

# Feature importance
print("\n" + "="*50)
print("FEATURE IMPORTANCE — PM ROADMAP")
print("="*50)
importance = pd.DataFrame({
    'feature':    feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)
importance['importance_pct'] = (importance['importance']*100).round(1)
print(importance.to_string(index=False))

MODEL PERFORMANCE
AUC:       0.997  (target: >0.90 for fraud)
F1 Score:  0.313  (target: >0.60)
Precision: 0.187  (of flagged txns, how many are real fraud?)
Recall:    0.964  (of all fraud, how many did we catch?)

CONFUSION MATRIX — PM TRANSLATION
True Negatives  (correctly said NOT fraud): 544,561
False Positives (wrongly blocked legit txn): 9,013
False Negatives (missed actual fraud):       78
True Positives  (correctly caught fraud):    2,067

BUSINESS IMPACT
Total fraudulent transactions in test: 2,145
Fraud caught:   2,067 (96.4%)
Fraud missed:   78 (3.6%)
Customers wrongly blocked: 9,013

Avg fraud amount: $528.36
Estimated fraud caught value: $1,092,113
Estimated fraud missed value: $41,212

FEATURE IMPORTANCE — PM ROADMAP
        feature  importance  importance_pct
            amt    0.593174       59.299999
     trans_hour    0.202882       20.299999
   category_enc    0.129494       12.900000
            age    0.044062        4.400000
       city_pop    0.020097        2.0

## Final results — PM conclusions

**Model scores**
- AUC: 0.997 — near-perfect fraud ranking
- Recall: 96.4% — catches almost all fraud
- Precision: 18.7% — aggressive flagging creates false alarms
- Fraud caught value: $1,092,113
- Fraud missed value: $41,212

**The core tradeoff**
9,013 legitimate customers wrongly blocked to catch 2,067 fraudsters.
This ratio reflects a business decision about brand vs protection.
A premium card brand would reduce the weight to lower false positives.
A protection-first card would keep it as is.

**Product roadmap from feature importance**
- P0: Amount trigger >$400 → push notification confirm (59% importance)
- P0: Night-time friction 10pm–2am → "did you make this?" (20% importance)
- P1: Stricter thresholds for online shopping category (13% importance)
- P2: False positive recovery flow — one-tap unblock to reduce 9,013 wrongly blocked customers